# Tome C3, Chapter 10: Beyond NP (Python companion)

Exercises first, solutions at the end. We evaluate quantified formulas, write Savitch's recursion as a formula and watch a single universal quantifier save an exponential, turn formula games into geography and play both, test the matching criterion for undirected geography, explore NFA universality, count models and perfect matchings, and finally reduce satisfiability to orthogonal vectors.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time
from collections import deque
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
random.seed(10)

## Exercises

**P10.1 (★) Evaluating quantified formulas.** Write a recursive evaluator for sentences $Q_1x_1\cdots Q_nx_n\,\chi$ with a CNF matrix, in the space-saving style of Theorem 10.4. For random 3-CNF matrices, compare the probability that $\exists x_1\cdots\exists x_n\,\chi$, $\exists\forall\exists\cdots\chi$ and $\forall x_1\cdots\forall x_n\,\chi$ are true, as the number of clauses grows.

**P10.2 (★★) One quantifier saves an exponential.** For a directed graph whose vertices are coded by $b$ bits, evaluate the formulas $\varphi_t(c,c')$ of Theorem 10.4 (reachability in at most $t$ steps), and check them against breadth-first search. Compare the size of $\varphi_t$ with the size of the naive formula $\psi_{2t}=\exists m\,\psi_t(c,m)\land\psi_t(m,c')$.

**P10.3 (★★) Formula games are geography.** Implement the reduction of Theorem 10.6 and a solver for generalised geography. On random formula games, check that E wins the formula game exactly when the first player wins the geography game.

**P10.4 (★★) Undirected geography and matchings.** Solve undirected vertex geography by brute force on random small graphs, and compare with the criterion of Exercise 10.4: the first player wins from $b$ if and only if $\nu(G-b)<\nu(G)$, where $\nu$ is the size of a maximum matching.

**P10.5 (★) Universality of NFAs.** Decide universality of NFAs by exploring the subset construction breadth-first, and return a shortest rejected word when there is one. How many subsets does the search visit for random NFAs, and for the NFAs $N_k$ that accept the words whose $k$-th letter from the end is not $a$?

**P10.6 (★) Counting models.** Count the models of random formulas, and check the parsimony claims of Exercise 10.8: the reduction of Theorem 9.5 can multiply the number of models, and the modified reduction preserves it.

**P10.7 (★★) Permanents.** Compute permanents by brute force over permutations, by Ryser's formula, and by dynamic programming over subsets of columns; count perfect matchings; check that the permanent and the determinant agree modulo $2$; recover the derangement numbers of Exercise 10.5. Time Ryser's formula against Gaussian elimination.

**P10.8 (★★) From satisfiability to orthogonal vectors.** Implement the reduction of Theorem 10.13, check it on random $k$-CNF formulas, and compare the naive quadratic algorithm for orthogonal vectors with the subset algorithm of Exercise 10.10 as the dimension varies.

## Solutions

### P10.1

In [ ]:
def eval_qbf(quants, clauses, n):
    # quants[i] in 'EA' for variable i+1; clauses over literals +-(i+1)
    assign = [None] * (n + 1)
    def clause_status():
        # False if some clause is already false; True if all are true
        all_true = True
        for c in clauses:
            sat, open_ = False, False
            for l in c:
                v = assign[abs(l)]
                if v is None:
                    open_ = True
                elif v == (l > 0):
                    sat = True; break
            if not sat:
                if not open_:
                    return False
                all_true = False
        return True if all_true else None
    def rec(i):
        st = clause_status()
        if st is not None:
            return st
        results = []
        for val in (False, True):
            assign[i] = val
            r = rec(i + 1)
            assign[i] = None
            if quants[i - 1] == 'E' and r:
                return True
            if quants[i - 1] == 'A' and not r:
                return False
        return quants[i - 1] == 'A'
    return rec(1)

def random_3cnf(n, m, rng):
    return [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), 3)] for _ in range(m)]

rng = random.Random(1)
n = 12
ms = [1, 2, 4, 8, 16, 24, 32, 48, 64]
prefixes = {'all exists (SAT)': 'E' * n, 'alternating, E first': ('EA' * n)[:n], 'all forall (tautology)': 'A' * n}
plt.figure(figsize=(6, 3.5))
for name, q in prefixes.items():
    probs = []
    for m in ms:
        probs.append(sum(eval_qbf(q, random_3cnf(n, m, rng), n) for _ in range(40)) / 40)
    plt.plot(ms, probs, 'o-', label=name)
    print(f"{name:24s}", dict(zip(ms, probs)))
plt.xlabel('clauses m (n = 12 variables)'); plt.ylabel('P(true)'); plt.legend(); plt.tight_layout(); plt.show()

A CNF matrix is true for all assignments only if it has no clauses at all (every clause with three distinct variables is falsified by some assignment), so the universal sentences die at once. The existential ones are satisfiable up to about $4.27n$ clauses (Chapter 9's notebook). The alternating ones sit in between: E must be able to answer every move of A, and this fails much earlier than plain satisfiability. The evaluator uses the space of one assignment and a recursion stack of depth $n$, as in the proof that $\textsc{Tqbf}\in\mathsf{PSPACE}$; the time is exponential.

### P10.2

In [ ]:
def random_digraph(b, p, rng):
    N = 2 ** b
    return {u: {v for v in range(N) if v != u and rng.random() < p} for u in range(N)}

def phi(G, t, c, c2, b, counter):
    # the formula of Theorem 10.4, evaluated directly (quantifiers over b-bit vertex codes)
    counter['evals'] += 1
    if t == 1:
        return c == c2 or c2 in G[c]
    N = 2 ** b
    for m in range(N):                                   # exists m
        ok = True
        for d in range(N):                               # forall d, e
            for e in range(N):
                if (d == c and e == m) or (d == m and e == c2):
                    if not phi(G, t // 2, d, e, b, counter):
                        ok = False; break
            if not ok:
                break
        if ok:
            return True
    return False

def bfs_dist(G, s):
    d, q = {s: 0}, deque([s])
    while q:
        x = q.popleft()
        for y in G[x]:
            if y not in d:
                d[y] = d[x] + 1; q.append(y)
    return d

rng = random.Random(2)
b = 3
for trial in range(3):
    G = random_digraph(b, 0.18, rng)
    for t in (1, 2, 4, 8):
        counter = {'evals': 0}
        agree = all(phi(G, t, c, c2, b, counter) == (bfs_dist(G, c).get(c2, 99) <= t)
                    for c in range(2 ** b) for c2 in range(2 ** b))
        print(f"graph {trial}, t = {t}: phi_t agrees with BFS on all pairs: {agree}  ({counter['evals']} evaluations)")

print("\nformula sizes, s = 100 bits per configuration, base formula of size s^2:")
s = 100
for k in [1, 5, 10, 20, 40]:
    savitch = s ** 2 + k * 10 * s
    naive_log10 = k * math.log10(2) + 2 * math.log10(s)
    print(f"t = 2^{k:2d}: quantified formula ~ {savitch:9d} symbols, naive formula ~ 10^{naive_log10:.1f} symbols")

The quantified formula agrees with breadth-first search on every pair, for every $t$. Its evaluation is slow (the universal quantifier over $(d,e)$ ranges over all pairs, of which only two matter), but its *size* grows by $O(s)$ per doubling of $t$, against a doubling of the naive formula. For configurations of $s$ bits and $t=2^{cs}$ steps, this is the difference between $O(s^2)$ and $2^{cs}$ symbols: the difference between a polynomial reduction and none at all.

### P10.3

In [ ]:
def geography_graph(clauses, n):
    arcs = {}
    def arc(u, v):
        arcs.setdefault(u, set()).add(v); arcs.setdefault(v, set())
    for i in range(1, n + 1):
        T, Lv, R, B = ('T', i), ('L', i), ('R', i), ('B', i)
        arc(T, Lv); arc(T, R); arc(Lv, B); arc(R, B)
        arc(B, ('T', i + 1) if i < n else 'c')
    for j, cl in enumerate(clauses):
        arc('c', ('cl', j))
        for l in cl:
            arc(('cl', j), ('R', l) if l > 0 else ('L', -l))
    return arcs

def first_player_wins(arcs, start):
    @lru_cache(maxsize=None)
    def wins(v, visited):
        # the player to move from v wins?
        return any(not wins(w, visited | frozenset([w])) for w in arcs[v] if w not in visited)
    return wins(start, frozenset([start]))

def formula_game(clauses, n):
    return eval_qbf(('EA' * n)[:n], clauses, n)

rng = random.Random(3)
agree, eWins = 0, 0
for trial in range(200):
    n = rng.choice([3, 5])
    m = rng.randint(1, 3 * n)
    cl = [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), rng.randint(1, 3))] for _ in range(m)]
    a = formula_game(cl, n)
    g = first_player_wins(geography_graph(cl, n), ('T', 1))
    agree += a == g; eWins += a
print(f"formula game and geography agree on {agree}/200 random games (E wins {eWins} of them)")

The two games agree everywhere. The geography solver knows nothing about formulas: it only explores who can move where. The diamonds force the players to "choose values", and the final clause gadget lets A challenge a clause and E answer with a literal that the play has made true. Generalised geography is just one of many $\mathsf{PSPACE}$-complete games, and the same diamond-and-challenge idea appears in the proofs for Go and Hex.

### P10.4

In [ ]:
def max_matching_size(vertices, edges):
    vertices = sorted(vertices)
    adj = {v: set() for v in vertices}
    for u, v in edges:
        if u in adj and v in adj:
            adj[u].add(v); adj[v].add(u)
    @lru_cache(maxsize=None)
    def best(remaining):
        if not remaining:
            return 0
        v = min(remaining)
        rest = remaining - {v}
        res = best(rest)                                      # v unmatched
        for w in adj[v]:
            if w in rest:
                res = max(res, 1 + best(rest - {w}))
        return res
    return best(frozenset(vertices))

def undirected_geography(n, edges, b):
    adj = {v: set() for v in range(n)}
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    @lru_cache(maxsize=None)
    def wins(v, visited):
        return any(not wins(w, visited | frozenset([w])) for w in adj[v] if w not in visited)
    return wins(b, frozenset([b]))

rng = random.Random(4)
agree = 0
for trial in range(300):
    n = rng.randint(2, 9)
    edges = [e for e in itertools.combinations(range(n), 2) if rng.random() < 0.35]
    b = rng.randrange(n)
    game = undirected_geography(n, edges, b)
    crit = max_matching_size(set(range(n)) - {b}, edges) < max_matching_size(set(range(n)), edges)
    agree += game == crit
print(f"matching criterion agrees with brute-force play on {agree}/300 random graphs")

The criterion is exact. The strategy behind it is simple to state and to play: if every maximum matching covers the start, the first player always moves to the partner of the current vertex in a fixed maximum matching; otherwise the second player does so in a maximum matching avoiding the start. One small change of rules, directed to undirected, moves the game from $\mathsf{PSPACE}$-complete to polynomial time.

### P10.5

In [ ]:
def nfa_universal(states, alphabet, delta, start, accept):
    # BFS over subsets; returns (True, visited) or (False, shortest rejected word, visited)
    s0 = frozenset(start) if isinstance(start, list) else frozenset([start])
    seen, q = {s0: ''}, deque([s0])
    while q:
        S = q.popleft()
        if not (S & accept):
            return False, seen[S], len(seen)
        for a in alphabet:
            T = frozenset(t for s in S for t in delta.get((s, a), ()))
            if T not in seen:
                seen[T] = seen[S] + a
                q.append(T)
    return True, None, len(seen)

def random_nfa(k, rng, p=0.25):
    alphabet = 'ab'
    delta = {(s, a): {t for t in range(k) if rng.random() < p} for s in range(k) for a in alphabet}
    accept = {0} | {s for s in range(k) if rng.random() < 0.8}
    return range(k), alphabet, delta, 0, accept

rng = random.Random(5)
print("random NFAs with 12 states:")
for trial in range(5):
    res = nfa_universal(*random_nfa(12, rng))
    print(f"  universal: {res[0]!s:5s}  shortest rejected word: {res[1]!r:12s}  subsets visited: {res[2]}")

def kth_from_end_not_a(k):
    # accepts the words of length < k, and those whose k-th letter from the end is b
    delta = {(0, 'a'): {0}, (0, 'b'): {0, 1}}
    for i in range(1, k):
        for x in 'ab':
            delta[(i, x)] = {i + 1}
    for i in range(k - 1):
        for x in 'ab':
            delta[(('s', i), x)] = {('s', i + 1)}
    states = list(range(k + 1)) + [('s', i) for i in range(k)]
    accept = {k} | {('s', i) for i in range(k)}
    return states, 'ab', delta, [0, ('s', 0)], accept

print("\nN_k: k-th letter from the end is not a")
for k in range(2, 13, 2):
    res = nfa_universal(*kth_from_end_not_a(k))
    print(f"  k = {k:2d}: {2 * k + 1:3d} states, universal: {res[0]!s:5s}, shortest rejected word: {res[1]!r}, subsets visited: {res[2]}")

Random NFAs are rarely interesting: either some short word is rejected, or the subsets quickly fill up. The NFAs $N_k$ are the classic hard case for the subset construction (Chapter 1): to know whether the $k$-th letter from the end is $a$, a deterministic automaton must remember the last $k$ letters, and the search visits about $2^k$ subsets before finding the rejected word $a$ followed by $k-1$ letters. Theorem 10.7 says that, unless $\mathsf P=\mathsf{PSPACE}$, no algorithm avoids such blow-ups in general.

### P10.6

In [ ]:
def count_models(clauses, n):
    cnt = 0
    for bits in itertools.product((False, True), repeat=n):
        if all(any(bits[abs(l) - 1] == (l > 0) for l in c) for c in clauses):
            cnt += 1
    return cnt

def split_clause(c, nxt):
    # reduction of Theorem 9.5 for one clause; nxt is the next free variable
    if len(c) <= 3:
        return [c], nxt
    k = len(c)
    z = list(range(nxt, nxt + k - 3))
    out = [[c[0], c[1], z[0]]]
    for i in range(1, k - 3):
        out.append([-z[i - 1], c[i + 1], z[i]])
    out.append([-z[-1], c[-2], c[-1]])
    return out, nxt + k - 3

def split_clause_parsimonious(c, nxt):
    if len(c) <= 3:
        return [c], nxt
    k = len(c)
    z = list(range(nxt, nxt + k - 2))         # z[0], ..., z[k-3]
    out = []
    def equiv_or(zv, u, v):                       # z <-> (u or v)
        return [[-zv, u, v], [zv, -u], [zv, -v]]
    out += equiv_or(z[0], c[0], c[1])
    for i in range(1, k - 2):
        out += equiv_or(z[i], z[i - 1], c[i + 1])    # z[i] <-> (l_1 or ... or l_{i+2})
    out.append([z[-1], c[-1]])
    return out, nxt + k - 2

rng = random.Random(6)
for trial in range(5):
    n = 6
    F = [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), rng.randint(4, 6))] for _ in range(3)]
    G, nxt = [], n + 1
    for c in F:
        cl, nxt = split_clause(c, nxt); G += cl
    H, nxt2 = [], n + 1
    for c in F:
        cl, nxt2 = split_clause_parsimonious(c, nxt2); H += cl
    print(f"models: original {count_models(F, n):3d}, Theorem 9.5 reduction {count_models(G, nxt - 1):4d}, "
          f"parsimonious version {count_models(H, nxt2 - 1):3d}")

The reduction of Theorem 9.5 preserves satisfiability but not the number of models: when a clause has several true literals, several values of the auxiliary variables work. Defining each auxiliary variable as a disjunction (here $z_1\leftrightarrow\ell_1\lor\ell_2$, $z_i\leftrightarrow z_{i-1}\lor\ell_{i+1}$, and the clause $z_{k-2}\lor\ell_k$, as in the solution of Exercise 10.8) makes them functions of the original variables, and the count is preserved exactly. Parsimonious reductions are what $\#\mathsf P$-completeness proofs are made of.

### P10.7

In [ ]:
def perm_brute(A):
    n = len(A)
    return sum(math.prod(A[i][s[i]] for i in range(n)) for s in itertools.permutations(range(n)))

def perm_ryser(A):
    n = len(A)
    total = 0
    for mask in range(1 << n):
        cols = [j for j in range(n) if mask >> j & 1]
        prod = 1
        for i in range(n):
            prod *= sum(A[i][j] for j in cols)
            if prod == 0:
                break
        total += (-1) ** (n - len(cols)) * prod
    return total

def perm_dp(A):
    # f[mask] = sum over assignments of the first |mask| rows to the columns in mask
    n = len(A)
    f = [0] * (1 << n); f[0] = 1
    for mask in range(1 << n):
        if f[mask] == 0:
            continue
        i = bin(mask).count('1')
        if i == n:
            continue
        for j in range(n):
            if not mask >> j & 1 and A[i][j]:
                f[mask | 1 << j] += f[mask] * A[i][j]
    return f[(1 << n) - 1]

def det_int(A):
    return round(np.linalg.det(np.array(A, dtype=float)))

rng = random.Random(7)
for trial in range(200):
    n = rng.randint(1, 7)
    A = [[rng.choice((0, 1, 1)) for _ in range(n)] for _ in range(n)]
    p = perm_brute(A)
    assert p == perm_ryser(A) == perm_dp(A)
    assert (p - det_int(A)) % 2 == 0
print("brute force, Ryser and subset DP agree on 200 random 0-1 matrices; perm = det mod 2 every time")

D = [perm_ryser([[0 if i == j else 1 for j in range(n)] for i in range(n)]) for n in range(1, 9)]
print("derangements D_1..D_8 via Ryser:", D)
print("n! * sum (-1)^i/i!           :", [round(math.factorial(n) * sum((-1) ** i / math.factorial(i) for i in range(n + 1))) for n in range(1, 9)])

ns, tr, td = [], [], []
for n in range(8, 17, 2):
    A = [[rng.choice((0, 1)) for _ in range(n)] for _ in range(n)]
    t0 = time.perf_counter(); perm_dp(A); tr.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); np.linalg.det(np.array(A, dtype=float)); td.append(time.perf_counter() - t0)
    ns.append(n)
    print(f"n = {n:2d}: permanent (subset DP) {tr[-1]:8.4f} s, determinant {td[-1] * 1e6:7.1f} µs")
plt.figure(figsize=(6, 3.5))
plt.semilogy(ns, tr, 'o-', label='permanent, $O(2^n n)$ dynamic programme')
plt.semilogy(ns, td, 's-', label='determinant, Gaussian elimination')
plt.xlabel('n'); plt.ylabel('seconds'); plt.legend(); plt.tight_layout(); plt.show()

Three methods give the same permanents; the determinant agrees with them modulo $2$ (Exercise 10.6), and Ryser's formula on $J-I$ produces the derangement numbers $0,1,2,9,44,265,1854,14833$. The timings show the gap between the two sums over permutations that differ only by signs: the determinant takes microseconds for any size here, the permanent doubles its time with each extra row. Valiant's theorem says this gap is no accident: unless counting problems collapse, no algorithm will make the permanent polynomial.

### P10.8

In [ ]:
def sat_to_ov(clauses, n):
    half = n // 2
    X1, X2 = range(1, half + 1), range(half + 1, n + 1)
    def vectors(vars_):
        vs = []
        for bits in itertools.product((False, True), repeat=len(vars_)):
            a = dict(zip(vars_, bits))
            vec = 0
            for j, c in enumerate(clauses):
                if not any(abs(l) in a and a[abs(l)] == (l > 0) for l in c):
                    vec |= 1 << j
            vs.append((bits, vec))
        return vs
    return vectors(X1), vectors(X2)

def ov_naive(A, B):
    for a in A:
        for b in B:
            if a & b == 0:
                return (a, b)
    return None

def ov_subsets(A, B, d):
    # table[T] = True if some vector of B is contained in T (as a set of coordinates)
    table = bytearray(1 << d)
    for b in B:
        table[b] = 1
    for i in range(d):
        bit = 1 << i
        for T in range(1 << d):
            if T & bit and table[T ^ bit]:
                table[T] = 1
    full = (1 << d) - 1
    return any(table[full & ~a] for a in A)

rng = random.Random(8)
agree = 0
for trial in range(100):
    n = rng.choice([6, 8, 10]); k = 3
    F = [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), k)] for _ in range(rng.randint(5, 6 * n))]
    VA, VB = sat_to_ov(F, n)
    found = ov_naive([v for _, v in VA], [v for _, v in VB]) is not None
    agree += found == (count_models(F, n) > 0)
print(f"OV instance has an orthogonal pair exactly when the formula is satisfiable: {agree}/100")

print("\nnaive O(N^2 d) against subset table O(2^d d + N d), N = 2000 vectors per side:")
N = 2000
for d in [8, 12, 16, 20]:
    A = [rng.getrandbits(d) | rng.getrandbits(d) for _ in range(N)]     # dense vectors: few orthogonal pairs
    B = [rng.getrandbits(d) | rng.getrandbits(d) for _ in range(N)]
    t0 = time.perf_counter(); r1 = ov_naive(A, B) is not None; t1 = time.perf_counter() - t0
    t0 = time.perf_counter(); r2 = ov_subsets(A, B, d); t2 = time.perf_counter() - t0
    assert r1 == r2
    print(f"d = {d:2d}: orthogonal pair: {r1!s:5s}  naive {t1:6.3f} s, subset table {t2:6.3f} s")

The reduction is exact: an orthogonal pair is a satisfying assignment cut in two halves. For the algorithms, the cost of the subset table, about $2^dd$, depends on the dimension alone, and that of the naive search on $N^2$; here, in Python, the table stops paying off around $d=16$ for $N=2000$. This is why the hardness of orthogonal vectors is a statement about dimensions somewhat larger than $\log N$. When the vectors are dense, orthogonal pairs are rare and the naive search runs to the end; Theorem 10.13 says that beating it by a polynomial factor, in the regime the reduction produces, would refute SETH.